In [0]:
%sql
-- Stage 1: Infrastructure
CREATE SCHEMA IF NOT EXISTS workspace.default;
CREATE VOLUME IF NOT EXISTS workspace.default.telco_data;

In [0]:
# Stage 2: Imports & Housekeeping
import gc
import xgboost as xgb
import numpy as np
import pandas as pd
import mlflow
from pyspark.sql import functions as F
from pyspark.ml.feature import StringIndexer, VectorAssembler

# Forcefully clear the Spark Connect ML Cache
try:
    spark.client.release_all() 
except:
    pass
gc.collect()

In [0]:
# Stage 3: Feature Engineering
df_raw = spark.table("silver_telco_churn")

# Create Interaction Features (Cost Velocity, Service Count, etc.)
df_enhanced = df_raw \
    .withColumn("Cost_Velocity", F.col("MonthlyCharges") / (F.col("tenure") + 1)) \
    .withColumn("Billing_Burden", F.col("TotalCharges") / (F.col("tenure") + 1))

service_cols = ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]
df_enhanced = df_enhanced.withColumn("Service_Count", 
    sum([F.when(F.col(c) == 'Yes', 1).otherwise(0) for c in service_cols]))

# Lineage Break to prevent 1GB Cache Overflow
df_enhanced = df_enhanced.localCheckpoint()

In [0]:
# Stage 4: Micro-Fitting & Splitting
categorical_cols = ["gender", "SeniorCitizen", "Partner", "Dependents", "PhoneService", "MultipleLines", 
                    "InternetService", "OnlineSecurity", "OnlineBackup", "DeviceProtection", 
                    "TechSupport", "StreamingTV", "StreamingMovies", "Contract", 
                    "PaperlessBilling", "PaymentMethod"]

numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges", "CLV", "CAC", "Cost_Velocity", "Billing_Burden", "Service_Count"]

# Manual Indexing to save memory
df_indexed = df_enhanced
for col in categorical_cols:
    df_indexed = StringIndexer(inputCol=col, outputCol=col + "_index", handleInvalid="keep").fit(df_indexed).transform(df_indexed)

df_indexed = StringIndexer(inputCol="Churn", outputCol="Churn_index", handleInvalid="error").fit(df_indexed).transform(df_indexed)

# Final Assembly
assembler_inputs = [c + "_index" for c in categorical_cols] + numeric_cols
df_transformed = VectorAssembler(inputCols=assembler_inputs, outputCol="features").transform(df_indexed).select("customerID", "features", "Churn_index").localCheckpoint()

train_df, test_df = df_transformed.randomSplit([0.8, 0.2], seed=42)

In [0]:
# Stage 5: Prepare for Native XGBoost
train_pd = train_df.toPandas()
test_pd = test_df.toPandas()

X_train = np.stack(train_pd['features'].apply(lambda x: x.toArray()))
y_train = train_pd['Churn_index']
X_test = np.stack(test_pd['features'].apply(lambda x: x.toArray()))
y_test = test_pd['Churn_index']

In [0]:
import xgboost as xgb
from sklearn.metrics import roc_auc_score

# 1. Convert Spark DataFrames to Pandas (safe for 7k rows)
train_pd = train_df.toPandas()
test_pd = test_df.toPandas()

# 2. Prepare the data for the Python-native XGBoost
# We pull the 'features' out of that suitcase we made earlier
import numpy as np
X_train = np.stack(train_pd['features'].apply(lambda x: x.toArray()))
y_train = train_pd['Churn_index']
X_test = np.stack(test_pd['features'].apply(lambda x: x.toArray()))
y_test = test_pd['Churn_index']

# 3. Train the "Python Native" XGBoost
print("Training Native XGBoost...")
xgb_model = xgb.XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1)
xgb_model.fit(X_train, y_train)

# 4. Score it
probs = xgb_model.predict_proba(X_test)[:, 1]
auc_score = roc_auc_score(y_test, probs)
print(f"-> Native XGBoost AUC: {auc_score:.4f}")

In [0]:
# Stage 7: Train the Tuned Champion
ratio = 5174 / 1869  # Fixing the imbalance
final_model = xgb.XGBClassifier(
    colsample_bytree=0.8, learning_rate=0.05, max_depth=3, n_estimators=100, 
    subsample=0.8, scale_pos_weight=ratio, eval_metric='auc', random_state=42
)
final_model.fit(X_train, y_train)

In [0]:
# Stage 7: Train the Tuned Champion (Run this to define 'final_xgb')
import xgboost as xgb

# 1. Use the ratio we calculated earlier to handle imbalance
ratio = 5174 / 1869 

# 2. Initialize the model with your Grid Search winners
final_xgb = xgb.XGBClassifier(
    colsample_bytree=0.8, 
    learning_rate=0.05, 
    max_depth=3, 
    n_estimators=100, 
    subsample=0.8, 
    scale_pos_weight=ratio, 
    eval_metric='auc', 
    random_state=42
)

# 3. Fit the model to your training data
# Note: Ensure you have already run the 'X_train' conversion cell!
print("Training the final champion model...")
final_xgb.fit(X_train, y_train)

print("Success! 'final_xgb' is now defined. You can now run your report cell.")

In [0]:
from sklearn.metrics import classification_report, roc_auc_score

# Now your existing code will work:
y_pred = final_xgb.predict(X_test)
print(classification_report(y_test, y_pred))

In [0]:
from sklearn.metrics import precision_recall_curve,precision_score,recall_score

# 1. Get the probabilities (the % chance of churn) instead of just Yes/No
# 'probs' was defined in your previous stage
# probs = final_xgb.predict_proba(X_test)[:, 1]

# 2. Test different thresholds: 0.3, 0.4, 0.5 (current), 0.6, 0.7
thresholds = [0.3,.38,.39, 0.4,.41,.42,.43,.44,.45,.46,.47,.48, 0.5, 0.6, 0.7]

print(f"{'Threshold':<12} | {'Precision':<10} | {'Recall':<10}")
print("-" * 40)

for t in thresholds:
    # Apply the custom threshold
    y_pred_custom = (probs >= t).astype(int)
    
    # Calculate metrics for the 'Churn' class (1.0)
    p = precision_score(y_test, y_pred_custom)
    r = recall_score(y_test, y_pred_custom)
    f1 = 2 * (p * r) / (p + r)
    
    print(f"{t:<12} | {p:<10.4f} | {r:<10.4f} | {f1:<10.4f}")

In [0]:
import matplotlib.pyplot as plt

# 1. Get importance scores from the champion model
# Match them to your final Stage 4 numeric_cols + categorical_indexers
all_features = [c + "_index" for c in categorical_cols] + numeric_cols
importance_scores = final_xgb.feature_importances_

# 2. Create a clean visualization
importance_df = pd.DataFrame({'Feature': all_features, 'Importance': importance_scores})
importance_df = importance_df.sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 8))
plt.barh(importance_df['Feature'].head(10), importance_df['Importance'].head(10), color='gold')
plt.xlabel('Importance Score')
plt.title('Top 10 Business Drivers of Churn')
plt.gca().invert_yaxis()
plt.show()

print("Stage 9 Complete: Insights generated.")

In [0]:
# 1. Apply your optimized 0.42 threshold
# 'probs' contains the raw probability (0.0 to 1.0) for each customer
custom_threshold = 0.42

predictions_df = pd.DataFrame({
    'customerID': test_pd['customerID'],
    'Churn_Probability': probs,
    # If prob >= 0.42, it's a 1 (Churn), otherwise 0 (Loyal)
    'Predicted_Churn': (probs >= custom_threshold).astype(int)
})

# 2. Convert to Spark and Save to the 'workspace' catalog
# Mode("overwrite") ensures that if you run this again, it updates the data
spark_predictions = spark.createDataFrame(predictions_df)
spark_predictions.write.mode("overwrite").saveAsTable("workspace.default.gold_churn_predictions")

print(f"Stage 10 Complete: Gold Predictions saved using a {custom_threshold} threshold.")

# 3. View the Top 10 customers most likely to leave
display(spark_predictions.sort("Churn_Probability", ascending=False).limit(10))

In [0]:
# Stage 11: Gen AI Retention Agent (Mock-up)
def generate_retention_email(customer_id, risk_score, top_reason):
    # This is where you would call an LLM API (like Google Gemini)
    prompt = f"""
    Customer {customer_id} has a {risk_score:.2f} churn risk.
    The top reason is {top_reason}. 
    Write a 2-sentence friendly email offering a relevant discount.
    """
    # For now, we'll simulate the AI output:
    return f"Subject: A special offer for you! | Logic: Addressed {top_reason}"

# Let's apply this to your top 3 at-risk customers
top_at_risk = spark.table("workspace.default.gold_churn_predictions").limit(3).toPandas()

print("Generating AI Retention Scripts...")
for index, row in top_at_risk.iterrows():
    email = generate_retention_email(row['customerID'], row['Churn_Probability'], "High Monthly Charges")
    print(f"--- Customer: {row['customerID']} ---\n{email}\n")

In [0]:
# Stage 12: Creating the Tableau Dashboard View
# We join the descriptive silver data with our ML predictions
sql_query = """
CREATE OR REPLACE VIEW workspace.default.v_churn_dashboard AS
SELECT 
    s.*,
    g.Churn_Probability,
    g.Predicted_Churn
FROM workspace.default.silver_telco_churn s
LEFT JOIN workspace.default.gold_churn_predictions g 
    ON s.customerID = g.customerID
"""
spark.sql(sql_query)
print("Success! Dashboard View created in 'workspace.default.v_churn_dashboard'.")